In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Python executable:", sys.executable)
print("All libraries imported successfully.")

Python executable: c:\Users\anton\OneDrive\Documents\PROG8245\240926\DataColletion-PreProcessing\.venv\Scripts\python.exe
All libraries imported successfully.


### Design choices

- **Relative file path:** The dataset is located inside the repository,
  allowing the notebook to run on another computer when opened from the
  project root.
- **DataFrame:** pandas provides a tabular structure suitable for inspecting,
  cleaning, and aggregating sales records.
- **Working copy:** `raw_sales` retains the full imported dataset, while
  `sales` contains an independent copy of its first 500 rows.
- **Selection limitation:** The first 500 rows satisfy the assignment's
  permitted selection method, but they are not a random sample.
- **Initial inspection:** Row and column counts confirm the loaded dimensions.
  The three-row preview shows the structure but does not establish data quality.
- **Source preservation:** This step only reads the CSV; it does not overwrite it.

In [2]:
from pathlib import Path

# Use a relative path so the notebook also works after cloning the repository.
data_path = Path("data") / "1000 Sales Records.csv"

# Load the original file and create an independent working copy.
raw_sales = pd.read_csv(data_path)
sales = raw_sales.head(500).copy()

print(f"Rows in the source file: {len(raw_sales):,}")
print(f"Rows selected for this project: {len(sales):,}")
print(f"Number of columns: {sales.shape[1]}")

sales.head(3)

Rows in the source file: 1,000
Rows selected for this project: 500
Number of columns: 14


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.20,263.33,3692591.20,2224085.18,1468506.02
1,North America,Canada,Vegetables,Online,M,11/7/2011,185941302,12/8/2011,3018,154.06,90.93,464953.08,274426.74,190526.34
2,Middle East and North Africa,Libya,Baby Food,Offline,C,10/31/2016,246222341,12/9/2016,1517,255.28,159.42,387259.76,241840.14,145419.62


## 2. Pick the Right Container

A dictionary suits each sales record because it provides access through
field names and allows values to be updated during cleaning, whereas a
namedtuple has fixed fields and does not support direct field reassignment.
A list can hold all sales records, while a set stores unique values,
such as the countries represented in the dataset.

In [3]:
# Convert the first row into a dictionary with column names as keys.
sample_record = sales.iloc[0].to_dict()

# Access individual values using their field names.
print("Container type:", type(sample_record).__name__)
print("Product category:", sample_record["Item Type"])
print("Unit price:", sample_record["Unit Price"])

# Display the complete record.
sample_record

Container type: dict
Product category: Cosmetics
Unit price: 437.2


{'Region': 'Middle East and North Africa',
 'Country': 'Libya',
 'Item Type': 'Cosmetics',
 'Sales Channel': 'Offline',
 'Order Priority': 'M',
 'Order Date': '10/18/2014',
 'Order ID': 686800706,
 'Ship Date': '10/31/2014',
 'Units Sold': 8446,
 'Unit Price': 437.2,
 'Unit Cost': 263.33,
 'Total Revenue': 3692591.2,
 'Total Cost': 2224085.18,
 'Total Profit': 1468506.02}

## 2. Pick the Right Container

A dictionary suits each sales record because it provides access through field names and allows values to be updated during cleaning, whereas a namedtuple has fixed fields and does not support direct field reassignment.
A list can hold all sales records, while a set stores unique values, such as the countries represented in the dataset.

In [4]:
# Convert the first row into a dictionary with column names as keys.
sample_record = sales.iloc[0].to_dict()

# Access individual values using their field names.
print("Container type:", type(sample_record).__name__)
print("Product category:", sample_record["Item Type"])
print("Unit price:", sample_record["Unit Price"])

# Display the complete record.
sample_record

Container type: dict
Product category: Cosmetics
Unit price: 437.2


{'Region': 'Middle East and North Africa',
 'Country': 'Libya',
 'Item Type': 'Cosmetics',
 'Sales Channel': 'Offline',
 'Order Priority': 'M',
 'Order Date': '10/18/2014',
 'Order ID': 686800706,
 'Ship Date': '10/31/2014',
 'Units Sold': 8446,
 'Unit Price': 437.2,
 'Unit Cost': 263.33,
 'Total Revenue': 3692591.2,
 'Total Cost': 2224085.18,
 'Total Profit': 1468506.02}

### 3. Code organization

The `SalesRecord` class and `build_sales_record()` function are implemented
in `src/sales_record.py` and imported into this notebook.
The module contains reusable logic, while the notebook documents the
workflow and displays results. The `src/__init__.py` file marks `src`
as a Python package.

In [5]:
from src.sales_record import build_sales_record

# Create a sales object using the reusable function.
first_sale = build_sales_record(sample_record)

print("Product category:", first_sale.data["Item Type"])
print(f"Calculated revenue: {first_sale.total():,.2f}")
print(f"Source revenue: {first_sale.data['Total Revenue']:,.2f}")

Product category: Cosmetics
Calculated revenue: 3,692,591.20
Source revenue: 3,692,591.20


## 4. Bulk Loaded

Convert the working DataFrame into a list of dictionaries, with one
dictionary per row. Then use `build_sales_record()` to create a
`SalesRecord` object for each dictionary.

The list preserves row order and retains repeated records so that
potential duplicates can be assessed during profiling.

In [6]:
# Convert each DataFrame row into a dictionary.
sales_dicts = sales.to_dict(orient="records")

# Build one SalesRecord object for each dictionary.
sales_records = [
    build_sales_record(record)
    for record in sales_dicts
]

# Check that the conversion preserved the number of records.
assert len(sales_records) == len(sales), "Record count mismatch."

print(f"Dictionaries created: {len(sales_dicts):,}")
print(f"SalesRecord objects created: {len(sales_records):,}")
print("First object type:", type(sales_records[0]).__name__)
print(f"First record revenue: {sales_records[0].total():,.2f}")

Dictionaries created: 500
SalesRecord objects created: 500
First object type: SalesRecord
First record revenue: 3,692,591.20


## 5. Quick Profiling

Calculate the minimum, mean, and maximum unit price across the 500
selected records. The mean is calculated per record, without weighting
by units sold.

Use a set to count distinct non-missing countries. The source contains
`Country`, not `shipping_city`, so this count describes countries only.
The assignment's city-field requirement remains unresolved.

In [7]:
# Summarize unit prices across the selected records.
price_summary = sales["Unit Price"].agg(["min", "mean", "max"])

# A set retains each distinct country value only once.
unique_countries = set(sales["Country"].dropna())

print(f"Minimum unit price: {price_summary['min']:,.2f}")
print(f"Mean unit price: {price_summary['mean']:,.2f}")
print(f"Maximum unit price: {price_summary['max']:,.2f}")
print(f"Unique countries: {len(unique_countries)}")

Minimum unit price: 9.33
Mean unit price: 274.30
Maximum unit price: 668.27
Unique countries: 171


## 6. Spot the Grime

Inspect missing values, duplicate rows, surrounding whitespace,
invalid prices or quantities, and invalid dates.

These checks measure potential data-quality issues without changing
the working dataset. A zero count means that the corresponding check
found no issues; it does not mean that the dataset is free of all errors.

In [8]:
# Identify text columns and count cells with surrounding whitespace.
text_columns = sales.select_dtypes(include=["object", "string"]).columns

whitespace_count = sum(
    int(
        (
            sales[column].notna()
            & sales[column].ne(sales[column].str.strip())
        ).sum()
    )
    for column in text_columns
)

# Convert values temporarily for validation without changing sales.
prices = pd.to_numeric(sales["Unit Price"], errors="coerce")
quantities = pd.to_numeric(sales["Units Sold"], errors="coerce")

order_dates = pd.to_datetime(
    sales["Order Date"], format="%m/%d/%Y", errors="coerce"
)
ship_dates = pd.to_datetime(
    sales["Ship Date"], format="%m/%d/%Y", errors="coerce"
)

# Each check states whether it counts cells or rows.
quality_report = pd.DataFrame(
    [
        ("Missing values", "cells", int(sales.isna().sum().sum())),
        ("Duplicate rows beyond the first", "rows",
         int(sales.duplicated().sum())),
        ("Surrounding whitespace", "cells", whitespace_count),
        ("Missing, non-numeric, or non-positive prices", "rows",
         int((prices.isna() | prices.le(0)).sum())),
        ("Missing, non-numeric, non-positive, or fractional quantities",
         "rows",
         int((
             quantities.isna()
             | quantities.le(0)
             | quantities.mod(1).ne(0)
         ).sum())),
        ("Missing or invalid order dates", "rows",
         int(order_dates.isna().sum())),
        ("Missing or invalid ship dates", "rows",
         int(ship_dates.isna().sum())),
        ("Shipping before purchase", "rows",
         int(ship_dates.lt(order_dates).sum())),
    ],
    columns=["Check", "Counting unit", "Issue count"],
)

quality_report

,Check,Counting unit,Issue count
0,Missing values,cells,0
1,Duplicate rows beyond the first,rows,0
2,Surrounding whitespace,cells,22
3,"Missing, non-numeric, or non-positive prices",rows,0
4,"Missing, non-numeric, non-positive, or fractio...",rows,0
5,Missing or invalid order dates,rows,0
6,Missing or invalid ship dates,rows,0
7,Shipping before purchase,rows,0


### Initial quality findings

The initial checks detected 22 text cells with leading or trailing
whitespace. The other seven checks reported zero issues.

The table below documents the affected fields and proposed corrections.
These are multiple occurrences of one issue type; they do not establish
three different categories of dirty data.

In [9]:
# Collect the locations and values of cells with surrounding whitespace.
whitespace_cases = []

for column in text_columns:
    values = sales[column]
    affected = values.notna() & values.ne(values.str.strip())

    for row_index, original_value in values[affected].items():
        whitespace_cases.append({
            "Row index": row_index,
            "Field": column,
            "Original value": repr(original_value),
            "Proposed value": repr(original_value.strip()),
        })

whitespace_details = pd.DataFrame(whitespace_cases)

print(f"Affected cells: {len(whitespace_details)}")
print(f"Affected rows: {whitespace_details['Row index'].nunique()}")

whitespace_details

Affected cells: 22
Affected rows: 22


,Row index,Field,Original value,Proposed value
0,29,Country,'Mauritius ','Mauritius'
1,78,Country,'Tunisia ','Tunisia'
2,159,Country,'Moldova ','Moldova'
3,180,Country,'Moldova ','Moldova'
4,199,Country,'Antigua and Barbuda ','Antigua and Barbuda'
5,211,Country,'Mauritius ','Mauritius'
6,235,Country,'Seychelles ','Seychelles'
7,242,Country,'Seychelles ','Seychelles'
8,248,Country,'Moldova ','Moldova'
9,271,Country,'Tunisia ','Tunisia'


## 7. Cleaning Rules

Apply `SalesRecord.clean()` to remove leading and trailing whitespace
from text fields. Preserve internal spaces, numeric values, and rows.

Create separate objects for cleaning so the original working DataFrame
remains available for comparison. Measure affected cells before and
after cleaning, and verify that the record count is unchanged.

In [10]:
# Create separate objects so earlier records remain unchanged.
cleaned_records = [
    build_sales_record(record).clean()
    for record in sales_dicts
]

# Convert the cleaned object dictionaries back into a DataFrame.
cleaned_sales = pd.DataFrame(
    [record.data for record in cleaned_records],
    index=sales.index,
)

# Count remaining cells with surrounding whitespace.
remaining_whitespace = sum(
    int(
        (
            cleaned_sales[column].notna()
            & cleaned_sales[column].ne(
                cleaned_sales[column].str.strip()
            )
        ).sum()
    )
    for column in text_columns
)

cleaning_comparison = pd.DataFrame({
    "Metric": ["Cells with surrounding whitespace", "Row count"],
    "Before": [len(whitespace_details), len(sales)],
    "After": [remaining_whitespace, len(cleaned_sales)],
})

assert remaining_whitespace == 0, "Whitespace issues remain."
assert len(cleaned_sales) == len(sales), "Record count changed."

cleaning_comparison

,Metric,Before,After
0,Cells with surrounding whitespace,22,0
1,Row count,500,500


## 8. Transformations

Convert `Order Date` and `Ship Date` from text into pandas datetime
columns using the source format: month/day/year.

Create a separate transformed DataFrame to preserve the cleaning
checkpoint. Use strict conversion so unexpected invalid dates raise
an error instead of being silently replaced with missing values.

Date conversion changes the data representation; it does not indicate
that the original date strings were incorrect.

In [11]:
# Preserve the cleaned data and create a copy for transformations.
transformed_sales = cleaned_sales.copy()

date_columns = ["Order Date", "Ship Date"]
types_before = cleaned_sales[date_columns].dtypes.astype(str)

# Parse the source dates using an explicit format.
for column in date_columns:
    transformed_sales[column] = pd.to_datetime(
        transformed_sales[column],
        format="%m/%d/%Y",
        errors="raise",
    )

# Show the change in data types.
type_comparison = pd.DataFrame({
    "Before": types_before,
    "After": transformed_sales[date_columns].dtypes.astype(str),
})

display(type_comparison)
transformed_sales[date_columns].head(3)

,Before,After
Order Date,str,datetime64[us]
Ship Date,str,datetime64[us]


,Order Date,Ship Date
0,2014-10-18,2014-10-31
1,2011-11-07,2011-12-08
2,2016-10-31,2016-12-09


## 9. Feature Engineering

Create two features from the parsed dates:

- `days_since_purchase`: days between the order date and a reference
  date defined as one day after the latest order in the selected dataset.
- `shipping_delay_days`: days between the order date and the ship date.
  This measures time until shipment, not delivery time.

Using a dataset-based reference date makes the results reproducible:
rerunning the notebook with the same input produces the same values.
These features are derived from existing dates, not externally observed.

In [12]:
# Keep the transformation checkpoint unchanged.
featured_sales = transformed_sales.copy()

# Use a reproducible reference date based on the selected dataset.
reference_date = (
    featured_sales["Order Date"].max() + pd.Timedelta(days=1)
)

# Calculate purchase age relative to the reference date.
featured_sales["days_since_purchase"] = (
    reference_date - featured_sales["Order Date"]
).dt.days

# Calculate the number of days between purchase and shipment.
featured_sales["shipping_delay_days"] = (
    featured_sales["Ship Date"] - featured_sales["Order Date"]
).dt.days

# Check that both features contain valid, non-negative values.
feature_columns = ["days_since_purchase", "shipping_delay_days"]

assert featured_sales[feature_columns].notna().all().all(), (
    "Missing values detected in date features."
)
assert featured_sales[feature_columns].ge(0).all().all(), (
    "Negative values detected in date features."
)

print(f"Reference date: {reference_date:%Y-%m-%d}")

featured_sales[
    ["Order Date", "Ship Date", *feature_columns]
].head(3)

Reference date: 2017-07-27


,Order Date,Ship Date,days_since_purchase,shipping_delay_days
0,2014-10-18,2014-10-31,1013,13
1,2011-11-07,2011-12-08,2089,31
2,2016-10-31,2016-12-09,269,39


## 10. Mini-Aggregation

Group the prepared records by country and sum their reported revenue.
Sort the results from highest to lowest and display the top ten countries.

Verify that grouping preserves the total revenue. The results describe only the selected 500 synthetic records, not actual market performance.
Country-level aggregation does not resolve the missing shipping-city field.

In [13]:
# Sum reported revenue for each country.
revenue_by_country = (
    featured_sales.groupby("Country", as_index=False)["Total Revenue"]
    .sum()
    .sort_values(
        ["Total Revenue", "Country"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

# Verify that no revenue was lost during aggregation.
total_revenue = featured_sales["Total Revenue"].sum()

assert np.isclose(
    revenue_by_country["Total Revenue"].sum(),
    total_revenue,
    rtol=0,
    atol=0.01,
), "Aggregated revenue does not match the source total."

# Calculate each country's percentage of the selected revenue.
revenue_by_country["revenue_share_pct"] = (
    revenue_by_country["Total Revenue"] / total_revenue * 100
)

display(
    revenue_by_country.head(10).style.format({
        "Total Revenue": "{:,.2f}",
        "revenue_share_pct": "{:.2f}",
    })
)

# Generate an insight directly from the computed results.
leader = revenue_by_country.iloc[0]

print(
    f"Within the selected 500 synthetic records, "
    f"{leader['Country']} has the highest reported revenue "
    f"at {leader['Total Revenue']:,.2f}, representing "
    f"{leader['revenue_share_pct']:.2f}% of the total."
)

,Country,Total Revenue,revenue_share_pct
0,Papua New Guinea,"15,629,197.78",2.20
1,Costa Rica,"15,578,320.59",2.19
2,Czech Republic,"13,653,048.70",1.92
3,Cuba,"13,277,423.33",1.87
4,Portugal,"12,548,165.49",1.76
5,United States of America,"11,927,294.75",1.68
6,South Africa,"11,874,129.63",1.67
7,Chad,"11,828,683.05",1.66
8,Tonga,"11,821,702.00",1.66
9,Swaziland,"11,816,973.90",1.66


Within the selected 500 synthetic records, Papua New Guinea has the highest reported revenue at 15,629,197.78, representing 2.20% of the total.


## 11. Serialization Checkpoint

Export the prepared transaction data to CSV and JSON without modifying the original source file.

Represent dates as ISO-formatted strings (`YYYY-MM-DD`) for consistent storage across both formats. Reload both files and compare their contents with the export table to verify that serialization preserved the data.

CSV and JSON do not preserve all pandas data types. Dates must be parsed again when loading these files for date calculations.

In [14]:
# Store generated files separately from the original CSV.
output_dir = Path("data") / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Format dates consistently without changing the analytical DataFrame.
export_sales = featured_sales.copy()

for column in date_columns:
    export_sales[column] = export_sales[column].dt.strftime("%Y-%m-%d")

csv_path = output_dir / "sales_preprocessed.csv"
json_path = output_dir / "sales_preprocessed.json"

# Export rows without the pandas index.
export_sales.to_csv(csv_path, index=False, encoding="utf-8")

# Store JSON as a list of records with readable indentation.
export_sales.to_json(
    json_path,
    orient="records",
    indent=2,
    force_ascii=False,
    double_precision=15,
)

# Reload both exports to verify their contents.
csv_reloaded = pd.read_csv(csv_path)
json_reloaded = pd.read_json(json_path, orient="records", convert_dates=False)

for reloaded in [csv_reloaded, json_reloaded]:
    pd.testing.assert_frame_equal(
        export_sales.reset_index(drop=True),
        reloaded.reset_index(drop=True),
        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=1e-8,
    )

print(f"CSV saved: {csv_path.as_posix()}")
print(f"JSON saved: {json_path.as_posix()}")
print(f"Verified rows per file: {len(export_sales):,}")
print("Both exports match the prepared data.")

CSV saved: data/processed/sales_preprocessed.csv
JSON saved: data/processed/sales_preprocessed.json
Verified rows per file: 500
Both exports match the prepared data.


## 12. Soft Interview Reflection

Functions helped me organize the workflow into reusable operations.
`build_sales_record()` created objects consistently from dictionaries, while `clean()` applied the same whitespace rule to every record.
The `total()` method kept the revenue calculation inside the class
representing each sale.

Separating these definitions into `src/sales_record.py` made the notebook easier to follow and avoided repeating implementation code. Assertions checked record counts, cleaning results, and exported data. This structure also makes future changes easier: I can update a method in one place and rerun the notebook to check its effects.

## Data Dictionary

This section combines the primary sales dataset's fields with documented fields from the Country Codes reference dataset.

Primary source: [Excel BI Analytics](https://excelbianalytics.com/downloads-18-sample-csv-files-data-sets-for-testing-sales/)

Secondary source: [Country Codes](https://github.com/datasets/country-codes)

Local reference data: `data/country_codes.csv`  
Local field definitions: `data/country_codes_metadata.yml`

Country identifiers describe countries; they do not identify customers, shipping cities, or promotional offers.

In [15]:
# Load reference fields as text to preserve country-code formatting.
country_metadata = pd.read_csv(
    Path("data") / "country_codes.csv",
    dtype="string",
    keep_default_na=False,
)

# Inspect the fields relevant to country identification.
metadata_columns = [
    "CLDR display name",
    "official_name_en",
    "ISO3166-1-Alpha-2",
    "ISO3166-1-Alpha-3",
]

print(f"Reference records: {len(country_metadata):,}")
country_metadata[metadata_columns].head(5)

Reference records: 249


,CLDR display name,official_name_en,ISO3166-1-Alpha-2,ISO3166-1-Alpha-3
0,Afghanistan,Afghanistan,AF,AFG
1,Åland Islands,Åland Islands,AX,ALA
2,Albania,Albania,AL,ALB
3,Algeria,Algeria,DZ,DZA
4,American Samoa,American Samoa,AS,ASM


### Country-name matching

Compare cleaned sales-country names with both the customary and official
English names in the reference dataset. Ignore capitalization and
surrounding whitespace during comparison.

List unmatched names for manual review. Do not guess country codes or
silently discard unmatched sales records.

In [16]:
# Collect both documented English-name variants from the reference data.
reference_names = pd.concat(
    [
        country_metadata["CLDR display name"],
        country_metadata["official_name_en"],
    ],
    ignore_index=True,
)

# Normalize names for comparison without changing the source columns.
reference_name_set = set(
    reference_names.str.strip().str.casefold()
)
reference_name_set.discard("")

sales_country_names = pd.Series(
    sorted(featured_sales["Country"].dropna().unique()),
    name="Country",
)

# Identify names that do not match either reference-name variant.
matched = (
    sales_country_names.str.strip().str.casefold()
    .isin(reference_name_set)
)

unmatched_countries = sales_country_names.loc[~matched].to_frame()

print(f"Distinct cleaned sales countries: {len(sales_country_names)}")
print(f"Matched country names: {int(matched.sum())}")
print(f"Names requiring review: {len(unmatched_countries)}")

unmatched_countries

Distinct cleaned sales countries: 171
Matched country names: 161
Names requiring review: 10


,Country
32,Cote d'Ivoire
36,Czech Republic
41,East Timor
48,Federated States of Micronesia
86,Macedonia
122,Republic of the Congo
144,Swaziland
152,The Bahamas
157,Turkey
163,United Kingdom


### Explicit country-name crosswalk

Some sales-country names differ from the reference dataset because of
historical names, spelling, or naming conventions.

The crosswalk below maps these source labels to ISO alpha-3 identifiers.
Original sales-country names are preserved. This supports reference-data
matching without claiming that historical labels were incorrect.

In [17]:
# Explicit mappings for the ten unmatched source labels.
country_aliases = {
    "Cote d'Ivoire": "CIV",
    "Czech Republic": "CZE",
    "East Timor": "TLS",
    "Federated States of Micronesia": "FSM",
    "Macedonia": "MKD",
    "Republic of the Congo": "COG",
    "Swaziland": "SWZ",
    "The Bahamas": "BHS",
    "Turkey": "TUR",
    "United Kingdom": "GBR",
}

alias_table = pd.DataFrame(
    country_aliases.items(),
    columns=["Sales country", "ISO3166-1-Alpha-3"],
)

# Retrieve the corresponding names from the downloaded reference.
alias_review = alias_table.merge(
    country_metadata[metadata_columns],
    on="ISO3166-1-Alpha-3",
    how="left",
    validate="many_to_one",
    indicator=True,
)

# Confirm that every proposed code exists in the reference.
assert alias_review["_merge"].eq("both").all(), (
    "Some country codes were not found in the reference."
)

# Confirm that all unmatched source labels have an explicit mapping.
assert set(unmatched_countries["Country"]) <= set(country_aliases), (
    "Some unmatched country names still need a mapping."
)

alias_review[
    [
        "Sales country",
        "official_name_en",
        "ISO3166-1-Alpha-2",
        "ISO3166-1-Alpha-3",
    ]
]

,Sales country,official_name_en,ISO3166-1-Alpha-2,ISO3166-1-Alpha-3
0,Cote d'Ivoire,Ivory Coast,CI,CIV
1,Czech Republic,Czechia,CZ,CZE
2,East Timor,Timor-Leste,TL,TLS
3,Federated States of Micronesia,Micronesia (Federated States of),FM,FSM
4,Macedonia,North Macedonia,MK,MKD
5,Republic of the Congo,Congo,CG,COG
6,Swaziland,Eswatini,SZ,SWZ
7,The Bahamas,Bahamas,BS,BHS
8,Turkey,Türkiye,TR,TUR
9,United Kingdom,United Kingdom of Great Britain and Northern I...,GB,GBR


### Synthetic fields required by the assignment

Copy existing transaction values into the required field names.
Generate fictional customer identifiers, coupon codes, and shipping
cities using fixed patterns for reproducibility.

These synthetic fields do not represent observed customers, promotions,
or destinations. The original source file remains unchanged.

In [18]:
# Extend the prepared data without changing earlier checkpoints.
final_sales = featured_sales.copy()

# Map existing values to the required field names.
final_sales["date"] = final_sales["Order Date"]
final_sales["product"] = final_sales["Item Type"]
final_sales["price"] = final_sales["Unit Price"]
final_sales["quantity"] = final_sales["Units Sold"]

# Generate reproducible fictional customer identifiers.
final_sales["customer_id"] = [
    f"CUST-{(i % 100) + 1:03d}"
    for i in range(len(final_sales))
]

# Assign synthetic promotions and fictional shipping locations.
coupon_options = ["NONE", "SAVE10", "SAVE20"]
city_options = ["Sample City A", "Sample City B", "Sample City C"]

final_sales["coupon_code"] = [
    coupon_options[i % len(coupon_options)]
    for i in range(len(final_sales))
]

final_sales["shipping_city"] = [
    city_options[i % len(city_options)]
    for i in range(len(final_sales))
]

required_fields = [
    "date", "customer_id", "product", "price",
    "quantity", "coupon_code", "shipping_city",
]

final_sales[required_fields].head(3)

,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,2014-10-18,CUST-001,Cosmetics,437.20,8446,NONE,Sample City A
1,2011-11-07,CUST-002,Vegetables,154.06,3018,SAVE10,Sample City B
2,2016-10-31,CUST-003,Baby Food,255.28,1517,SAVE20,Sample City C


### Synthetic city profiling and coupon transformation

Count distinct fictional shipping cities using a set.
Convert synthetic coupon codes into numeric discount rates:
`NONE` = 0%, `SAVE10` = 10%, and `SAVE20` = 20%.

These rates are exercise assumptions, not observed promotions.
The original reported revenue remains unchanged.

In [19]:
# Count distinct synthetic shipping cities.
unique_cities = set(final_sales["shipping_city"])

# Translate each synthetic coupon into a numeric discount rate.
discount_rates = {
    "NONE": 0.0,
    "SAVE10": 0.10,
    "SAVE20": 0.20,
}

final_sales["discount_rate"] = (
    final_sales["coupon_code"].map(discount_rates)
)

assert final_sales["discount_rate"].notna().all(), (
    "An unrecognized coupon code was found."
)

print(f"Unique synthetic shipping cities: {len(unique_cities)}")

final_sales[
    ["shipping_city", "coupon_code", "discount_rate"]
].head(3)

Unique synthetic shipping cities: 3


,shipping_city,coupon_code,discount_rate
0,Sample City A,NONE,0.0
1,Sample City B,SAVE10,0.1
2,Sample City C,SAVE20,0.2


### Three controlled synthetic cleaning cases

Create three demonstration records with deliberately introduced issues:
surrounding whitespace, a missing coupon, and inconsistent coupon casing.

For these synthetic examples, a missing coupon means no promotion and
is replaced with `NONE`. Cleaning is performed inside `clean()`.
These examples are separate from the 500 transaction records.

In [20]:
from src.sales_record import SalesRecord


class SyntheticSalesRecord(SalesRecord):
    """Extend the existing class with synthetic coupon cleaning rules."""

    def clean(self):
        """Trim text, fill missing coupons, and standardize coupon casing."""
        super().clean()

        coupon = self.data.get("coupon_code")

        if pd.isna(coupon) or coupon == "":
            self.data["coupon_code"] = "NONE"
        else:
            self.data["coupon_code"] = coupon.upper()

        return self


# Create separate examples and deliberately introduce three issue types.
examples = final_sales.head(3).to_dict(orient="records")
examples[0]["shipping_city"] = " Sample City A "
examples[1]["coupon_code"] = None
examples[2]["coupon_code"] = "save20"


def count_demo_issues(records):
    """Count the three issue types in the demonstration records."""
    return [
        sum(
            row["shipping_city"] != row["shipping_city"].strip()
            for row in records
        ),
        sum(pd.isna(row["coupon_code"]) for row in records),
        sum(
            isinstance(row["coupon_code"], str)
            and row["coupon_code"] != row["coupon_code"].upper()
            for row in records
        ),
    ]


cleaned_examples = [
    SyntheticSalesRecord(row).clean().data
    for row in examples
]

demo_comparison = pd.DataFrame({
    "Issue": [
        "Surrounding city whitespace",
        "Missing coupon",
        "Inconsistent coupon casing",
    ],
    "Before": count_demo_issues(examples),
    "After": count_demo_issues(cleaned_examples),
})

assert demo_comparison["After"].eq(0).all(), "Cleaning issues remain."

demo_comparison

,Issue,Before,After
0,Surrounding city whitespace,1,0
1,Missing coupon,1,0
2,Inconsistent coupon casing,1,0


### Consolidated Data Dictionary

Sources:
- **P:** Excel BI Analytics sales CSV; field meanings interpreted from its headers.
- **S:** Country Codes CSV and its accompanying `country_codes_metadata.yml`.
- **G:** Deterministically generated synthetic values.
- **D:** Derived from existing fields.

The transaction fields below belong to `final_sales`. The last four
fields document the secondary reference table inspected in this notebook;
they are not added to the transaction exports.

| Field | Type | Description | Source |
|---|---|---|---|
| Region | string | Sales region label supplied by the source. | P |
| Country | string | Source country label, with surrounding whitespace removed. | P |
| Item Type | string | Product category. | P |
| Sales Channel | string | Online or offline sales channel. | P |
| Order Priority | string | Source order-priority code. | P |
| Order Date | datetime | Order date parsed from month/day/year text. | P + D |
| Order ID | integer | Source order identifier; not a customer identifier. | P |
| Ship Date | datetime | Shipment date parsed from month/day/year text. | P + D |
| Units Sold | integer | Number of units sold in the record. | P |
| Unit Price | float | Selling price per unit; currency is not established here. | P |
| Unit Cost | float | Cost per unit reported by the source. | P |
| Total Revenue | float | Reported revenue for the sales record. | P |
| Total Cost | float | Reported total cost for the sales record. | P |
| Total Profit | float | Reported profit for the sales record. | P |
| days_since_purchase | integer | Reference date minus Order Date, in days; reference is 2017-07-27. | D |
| shipping_delay_days | integer | Ship Date minus Order Date, in days; not delivery duration. | D |
| date | datetime | Copy of Order Date for the required transaction schema. | P + D |
| product | string | Copy of Item Type. | P + D |
| price | float | Copy of Unit Price. | P + D |
| quantity | integer | Copy of Units Sold. | P + D |
| customer_id | string | Fictional customer ID cycling through 100 identifiers. | G |
| coupon_code | string | Fictional promotion cycling through NONE, SAVE10 and SAVE20. | G |
| shipping_city | string | Fictional location cycling through Sample City A, B and C. | G |
| discount_rate | float | Coupon mapping: NONE = 0, SAVE10 = 0.10, SAVE20 = 0.20. | G + D |
| CLDR display name | string | Customary English country name in the reference. | S |
| official_name_en | string | Official English country or area name in the reference. | S |
| ISO3166-1-Alpha-2 | string | Two-letter country identifier. | S |
| ISO3166-1-Alpha-3 | string | Three-letter country identifier. | S |

Dates are serialized as YYYY-MM-DD strings in CSV and JSON.
The country crosswalk documents naming correspondences between sources.
Synthetic fields and controlled cleaning examples are educational
additions, not observations from the original sales data.

In [21]:
# Export all required fields, including the synthetic additions.
final_export = final_sales.copy()

for column in ["Order Date", "Ship Date", "date"]:
    final_export[column] = final_export[column].dt.strftime("%Y-%m-%d")

final_export.to_csv(csv_path, index=False, encoding="utf-8")
final_export.to_json(
    json_path,
    orient="records",
    indent=2,
    force_ascii=False,
    double_precision=15,
)

# Reload with explicit text handling for coupon codes such as NONE.
csv_check = pd.read_csv(csv_path, keep_default_na=False)
json_check = pd.read_json(
    json_path, orient="records", convert_dates=False
)

for reloaded in [csv_check, json_check]:
    pd.testing.assert_frame_equal(
        final_export.reset_index(drop=True),
        reloaded.reset_index(drop=True),
        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=1e-8,
    )

assert len(final_export) == 500
assert set(required_fields).issubset(final_export.columns)

print(f"Final exports verified: {len(final_export)} rows.")
print(f"Columns exported: {len(final_export.columns)}")
print("All required transaction fields are present.")

Final exports verified: 500 rows.
Columns exported: 24
All required transaction fields are present.
